**1st part: dataloader, loss function, and training function**




In [ ]:
!pip install --upgrade gdown

In [ ]:
from google.colab import drive
import os

In_this_space_flag = True

if In_this_space_flag == True and os.path.isdir("/content/oxford-iiit-pet/") == False:
  # 1 do everything in this temporary space
  os.system("gdown https://drive.google.com/uc?id=1ymthIy1mSTw0RFhTCd9NCmA-pJi6zjtw")
  os.system("unzip oxford-iiit-pet.zip")
  os.remove("oxford-iiit-pet.zip")
else:
  drive.mount('/content/drive')
  assert os.path.isdir("/content/drive/My Drive/oxford-iiit-pet/") == True

In [ ]:
from __future__ import print_function, division
from xml.dom import HierarchyRequestErr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy
import cv2
import shutil
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image
import random


class OxfordPetDataset(torch.utils.data.Dataset):
    def __init__(self, data_list, num_class, file_root, mode, transform=None):
        data_dic = {str(num):[] for num in  range(num_class)}

        images = []
        labels = open(data_list).readlines()
        for line in labels:
            items = line.strip('\n').split()
            img_name = items[0]
            label = str(int(items[1]) - 1)

            if int(label) > 23:
                continue

            data_dic[label].append(img_name)

        for cls in range(num_class):
            file_list = data_dic[str(cls)]
            file_num = len(file_list)
            imgs_list = [(file_root + file_list[i] + '.jpg', cls) for i in range(file_num)]
            images = images + imgs_list

        self.images = images
        self.transform = transform


    def __getitem__(self, index):
        img_name, label = self.images[index]
        assert os.path.exists(img_name) == True
        img = Image.open(img_name).convert('RGB')

        if self.transform is not None:
            img = self.transform(img)

        # Convert label to tensor:  Alex 2024/08/29, Without converting to torch.long might not throw any error in this example but it is a good practice to always do the conversion.
        label = torch.tensor(label, dtype=torch.long)
        return (img, label) if label is not None else img

    def __len__(self):
        return len(self.images)

def OxfordPet_dataloader(file_root, data_batch, train_data_list, val_data_list, num_class, img_size):

    data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(img_size),
        #transforms.RandomResizedCrop(224, scale=(0.9,1.0)),
        transforms.CenterCrop(img_size),
        #transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        transforms.RandomRotation(degrees=5),
        #transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        #transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
    'val': transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        #transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
    }

    train_set = OxfordPetDataset(train_data_list, num_class, file_root, 'train', data_transforms['train'])
    val_set = OxfordPetDataset(val_data_list, num_class,file_root, 'val',data_transforms['val'])


    image_datasets = {'train': train_set , 'val': val_set }
    dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=data_batch, shuffle=True, num_workers=8) for x in ['train', 'val']}
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}

    return {'dataloaders':dataloaders, 'dataset_sizes':dataset_sizes}




def train_model(model, criterion, optimizer, scheduler, num_epochs, dataloaders, dataset_sizes, device, num_class, save_model_flag=False ):

    best_acc = 0.0

    for epoch in range(num_epochs):

        print('Epoch {}/{}'.format(epoch, num_epochs - 1))


        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
                print('start training!')
                since_training = time.time()
            else:
                model.eval()   # Set model to evaluation mode
                print('start evaluation!')
                since_val = time.time()

            running_loss = 0.0
            running_corrects = 0


            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)

                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            if phase == 'train':
                final_train_acc = epoch_acc

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                epoch_on_best_acc = epoch

            if phase == 'val':
                final_val_acc = epoch_acc
                print('{} Loss: {:.4f} Acc: {:.4f} best_acc: {:.4f}'.format(phase, epoch_loss, epoch_acc, best_acc))

            if phase == 'train':
                time_elapsed = time.time() - since_training
                time_elapsed_mins = time_elapsed // 60
                #print('Training a epoch for {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))
            else:
                time_elapsed = time.time() - since_val
                time_elapsed_mins = time_elapsed // 60
                #print('Validating a epoch for {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))


    return {'final epoch train accuracy': final_train_acc.cpu().detach().numpy(), 'final epoch val accuracy': final_val_acc.cpu().detach().numpy(),\
     'highest val accuracy': best_acc.cpu().detach().numpy(), 'highest val accuracy epoch': epoch_on_best_acc, 'training time in mins': time_elapsed_mins}



Main training function

In [ ]:
#import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import time
import os
import copy
import cv2
import random
from decimal import Decimal
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from collections import OrderedDict


last_highest_val_accuracy = 0

# check if GPU is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print('torch.cuda.is_available()={}'.format(torch.cuda.is_available()))



if In_this_space_flag == True:
  ## version 1: download file
  # data folder
  train_data_list = '/content/oxford-iiit-pet/file_train_list.txt'
  val_data_list = '/content/oxford-iiit-pet/file_val_list.txt'
  file_root = '/content/oxford-iiit-pet/images/'
else:
  ## version 2: access the folder in your google drive
  train_data_list = '/content/drive/My Drive/oxford-iiit-pet/file_train_list.txt'
  val_data_list = '/content/drive/My Drive/oxford-iiit-pet/file_val_list.txt'
  file_root = '/content/drive/My Drive/oxford-iiit-pet/images/'



num_class = 24#37
img_size = 224

num_epochs = 10
lr = 0.0002
num_batch = 12

## 2. data preparation
data_dic = OxfordPet_dataloader(file_root, num_batch, train_data_list, val_data_list, num_class, img_size)

## 3. define model: the model will be downloaded to a temp folder
model_ft = models.resnet18(pretrained=True)

## 4. some surgery for the pretrained model
model_ft.fc = nn.Sequential(
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, num_class))

#print(model_ft)
model_ft = model_ft.to(device)

## 5. define criterion
criterion = nn.CrossEntropyLoss()

## 6. define solver
#optimizer = optim.SGD(model_ft.parameters(), lr=cfg['lr'], momentum=cfg['momentum'], weight_decay=0.0001)
optimizer = optim.Adam(model_ft.parameters(), lr=lr)


## 7. define learning rate decay policy: step size & gamma
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 8. training the defined model for the designated epochs
result_dic = train_model(model_ft, criterion, optimizer, exp_lr_scheduler, num_epochs=num_epochs, dataloaders = data_dic['dataloaders']\
, dataset_sizes = data_dic['dataset_sizes'], device = device, num_class=num_class)

highest_val_accuracy = result_dic['highest val accuracy'].round(4)

print('In {} epochs, highest val accuracy={} is achieved at epoch-{}'.format(num_epochs, highest_val_accuracy, result_dic['highest val accuracy epoch'] ) )




torch.cuda.is_available()=True


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]


Epoch 0/9
start training!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


start evaluation!
val Loss: 0.5176 Acc: 0.8446 best_acc: 0.8446
Epoch 1/9
start training!
start evaluation!
val Loss: 0.4954 Acc: 0.8395 best_acc: 0.8446
Epoch 2/9
start training!
start evaluation!
val Loss: 0.5666 Acc: 0.8324 best_acc: 0.8446
Epoch 3/9
start training!
start evaluation!
val Loss: 0.4361 Acc: 0.8617 best_acc: 0.8617
Epoch 4/9
start training!
start evaluation!
val Loss: 0.5473 Acc: 0.8352 best_acc: 0.8617
Epoch 5/9
start training!
start evaluation!
val Loss: 0.2950 Acc: 0.9062 best_acc: 0.9062
Epoch 6/9
start training!
start evaluation!
val Loss: 0.2660 Acc: 0.9133 best_acc: 0.9133
Epoch 7/9
start training!
start evaluation!
val Loss: 0.2509 Acc: 0.9183 best_acc: 0.9183
Epoch 8/9
start training!
start evaluation!
val Loss: 0.2426 Acc: 0.9176 best_acc: 0.9183
Epoch 9/9
start training!
start evaluation!
val Loss: 0.2528 Acc: 0.9234 best_acc: 0.9234
In 10 epochs, highest val accuracy=0.9234 is achieved at epoch-9


In 10 epochs, highest val accuracy=0.9212 is achieved at epoch-9